# BayesRTMMRL C-branch sharpening 机制验证

目的：严谨验证 **BayesRTMMRL 的 C branch ECE 增大** 到底来自哪里，而不是只看表面 `acc / ECE / confidence`。

验证问题：

1. **BayesRT C 的 overconfidence 是 posterior mean path 已经存在，还是 T MC sampling 造成？**
   - `C_Tmean`: text posterior mean，不采样 T。
   - `C_Tmc`: text posterior MC sampling。

2. **T posterior 是否产生有效 epistemic disagreement？**
   - `sample_agreement`
   - `variation_ratio`
   - `vote_entropy`
   - `predictive_entropy`
   - `expected_entropy`
   - `mutual_information`
   - `confidence_variance`
   - `margin_variance`

3. **ECE 增量来自哪些 confidence bins / 哪类样本？**
   - reliability bins
   - high-confidence wrong samples
   - paired outcome: `both_correct / harmful_flip / beneficial_flip / both_wrong`

该 notebook 默认 **不修改源码、不训练模型**，只加载已有 MMRL 与 BayesRTMMRL checkpoint，并在 notebook 内部重构 BayesRT 的 forward components。


In [1]:
# =========================
# 0. 配置区：按你的服务器路径修改
# =========================
from pathlib import Path

REPO_ROOT = Path("/root/autodl-tmp/MMRL")
DATASET_ROOT = Path("/root/autodl-tmp/MMRL/DATASETS")

CASE = {
    "case_name": "dtd_16shot_seed1",
    "dataset": "dtd",
    "protocol": "FS",
    "shot": 16,
    "seed": 1,

    # 如果你之前的验证 notebook 已经能跑，直接复制那里的配置。
    "dataset_config_file": "configs/datasets/dtd.yaml",
    "method_config_file_mmrl": "",
    "method_config_file_bayesrt": "",
    "protocol_config_file": "",
    "runtime_config_file": "",
    "exp_config": "",

    "mmrl_model_dir": "/root/autodl-tmp/MMRL/output_refactor/MMRL/FS/fewshot_train/dtd/shots_16/ViT-B-16/default/seed1",
    "bayesrt_model_dir": "/root/autodl-tmp/MMRL/output_refactor/BayesRTMMRL/FS/fewshot_train/dtd/shots_16/ViT-B-16/default/seed1",

    "load_epoch": 50,
    "split": "test",
}

OUT_DIR = REPO_ROOT / "output_refactor" / "analysis" / "bayesrt_c_branch_mechanism"
OUT_DIR.mkdir(parents=True, exist_ok=True)

N_BINS = 15
HC_THRESHOLDS = [0.80, 0.90, 0.95]
N_MC_OVERRIDE = None
SAVE_SAMPLE_DIAGNOSTICS = True


# 优先从训练目录日志恢复 run.py 的 Arguments。
# 这是为了避免 MODEL.BACKBONE.NAME / dataset-config / runtime-config 等漏填。
AUTO_RESTORE_ARGS_FROM_LOG = True

# 如果日志不可用，可以在这里手动补充关键 cfg opts。
# 例如：
# MANUAL_COMMON_OPTS = ["MODEL.BACKBONE.NAME", "ViT-B/16"]
MANUAL_COMMON_OPTS = []
MANUAL_MMRL_OPTS = []
MANUAL_BAYESRT_OPTS = []


In [2]:
# =========================
# 1. 环境初始化
# =========================
import os
import sys
import json
import math
import copy
import importlib
from argparse import Namespace
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("TORCH_NUM_THREADS", "1")
os.environ.setdefault("TORCH_NUM_INTEROP_THREADS", "1")

REPO_ROOT = Path(REPO_ROOT)
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from dassl.engine import build_trainer
from dassl.utils import set_random_seed
from core.config import setup_cfg
from core.types import EvalContext
from core.utils import import_optional_modules

import_optional_modules([
    "datasets.oxford_pets", "datasets.oxford_flowers", "datasets.fgvc_aircraft",
    "datasets.dtd", "datasets.eurosat", "datasets.stanford_cars", "datasets.food101",
    "datasets.sun397", "datasets.caltech101", "datasets.ucf101", "datasets.imagenet",
    "datasets.imagenetv2", "datasets.imagenet_sketch", "datasets.imagenet_a", "datasets.imagenet_r",
])

importlib.import_module("trainers.refactor_runner")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device =", device)

try:
    import subprocess
    git_head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_ROOT).decode().strip()
except Exception as e:
    git_head = f"unknown: {e}"
print("git_head =", git_head)


device = cuda
git_head = 36bc2de79e99e4b28cc368b101542bba7ffc115d


In [3]:

# =========================
# 2. 构建 trainer / 加载 checkpoint
# =========================
import ast
import glob

def abs_or_empty(path_like):
    if not path_like:
        return ""
    p = Path(path_like)
    if p.is_absolute():
        return str(p)
    return str((REPO_ROOT / p).resolve())


def _coerce_arg_value(key, value):
    value = str(value).strip()
    if key in {"seed", "load_epoch"}:
        if value in {"None", "null", ""}:
            return None
        try:
            return int(value)
        except Exception:
            return -1 if key == "seed" else None

    if key in {"eval_only", "no_train"}:
        return value.lower() in {"true", "1", "yes"}

    if key == "opts":
        if value in {"None", "null", ""}:
            return []
        try:
            parsed = ast.literal_eval(value)
            if parsed is None:
                return []
            if isinstance(parsed, (list, tuple)):
                return [str(x) for x in parsed]
        except Exception:
            pass
        return []

    return value


def find_training_log(model_dir):
    model_dir = Path(model_dir)
    roots = [
        model_dir,
        model_dir.parent,
        model_dir / "refactor_model",
    ]
    names = [
        "log.txt",
        "log.txt-*",
        "run.log",
        "stdout.log",
        "*.log",
    ]

    candidates = []
    for root in roots:
        if not root.exists():
            continue
        for name in names:
            candidates.extend(root.glob(name))

    # Some runs store logs one level above model dir.
    try:
        for p in model_dir.parents:
            if p == REPO_ROOT or len(str(p)) < len(str(REPO_ROOT)):
                break
            for name in ["log.txt", "log.txt-*", "*.log"]:
                candidates.extend(p.glob(name))
    except Exception:
        pass

    candidates = [p for p in candidates if p.is_file()]
    candidates = sorted(set(candidates), key=lambda p: p.stat().st_mtime, reverse=True)
    return candidates[0] if candidates else None


def parse_args_from_log(log_path):
    """
    Parse run.py print_args() block:
        ** Arguments **
        key: value
        ...
        ** Config **
    """
    log_path = Path(log_path)
    text = log_path.read_text(encoding="utf-8", errors="ignore")
    lines = text.splitlines()

    start = None
    end = None
    for i, line in enumerate(lines):
        if "** Arguments **" in line:
            start = i + 1
            break
    if start is None:
        return {}

    for j in range(start, len(lines)):
        if "** Config **" in lines[j]:
            end = j
            break
    if end is None:
        end = min(len(lines), start + 120)

    args = {}
    for line in lines[start:end]:
        line = line.strip()
        if not line or line.startswith("*"):
            continue
        if ": " not in line:
            continue
        key, value = line.split(": ", 1)
        key = key.strip().replace("-", "_")
        args[key] = _coerce_arg_value(key, value)

    return args


def make_args_from_case(case, method_name, method_config_file, model_dir):
    # Default manual construction.
    opts = [
        "METHOD.NAME", method_name,
        "METHOD.EXEC_MODE", "online",
        "DATASET.NUM_SHOTS", str(case["shot"]),
    ]
    opts += list(MANUAL_COMMON_OPTS)
    opts += list(MANUAL_MMRL_OPTS if method_name == "MMRL" else MANUAL_BAYESRT_OPTS)

    return Namespace(
        root=str(DATASET_ROOT),
        output_dir="",
        dataset_config_file=abs_or_empty(case.get("dataset_config_file", "")),
        method_config_file=abs_or_empty(method_config_file),
        protocol_config_file=abs_or_empty(case.get("protocol_config_file", "")),
        runtime_config_file=abs_or_empty(case.get("runtime_config_file", "")),
        exp_config=abs_or_empty(case.get("exp_config", "")),
        method=method_name,
        protocol=str(case.get("protocol", "FS")),
        exec_mode="online",
        seed=int(case["seed"]),
        trainer="RefactorRunner",
        eval_only=True,
        model_dir=str(model_dir),
        load_epoch=case.get("load_epoch", None),
        no_train=True,
        opts=opts,
    )


def make_args_from_log_or_case(case, method_name, method_config_file, model_dir):
    base = make_args_from_case(case, method_name, method_config_file, model_dir)

    if not bool(AUTO_RESTORE_ARGS_FROM_LOG):
        return base, None, {}

    log_path = find_training_log(model_dir)
    if log_path is None:
        print(f"[WARN] no log found under {model_dir}; using manual CASE config.")
        return base, None, {}

    parsed = parse_args_from_log(log_path)
    if not parsed:
        print(f"[WARN] failed to parse Arguments from {log_path}; using manual CASE config.")
        return base, log_path, {}

    args = vars(base).copy()

    # Restore only run.py argument names. Keep eval_only/no_train/model_dir/load_epoch from current CASE.
    restore_keys = [
        "root",
        "output_dir",
        "dataset_config_file",
        "method_config_file",
        "protocol_config_file",
        "runtime_config_file",
        "exp_config",
        "method",
        "protocol",
        "exec_mode",
        "seed",
        "trainer",
        "opts",
    ]

    for k in restore_keys:
        if k in parsed:
            args[k] = parsed[k]

    # Force method/model_dir/load_epoch for the model we are currently loading.
    args["method"] = method_name
    args["model_dir"] = str(model_dir)
    args["load_epoch"] = case.get("load_epoch", args.get("load_epoch", None))
    args["eval_only"] = True
    args["no_train"] = True

    # Absolute path normalization.
    for k in ["dataset_config_file", "method_config_file", "protocol_config_file", "runtime_config_file", "exp_config"]:
        args[k] = abs_or_empty(args.get(k, ""))

    # Ensure opts list and append manual overrides.
    opts = args.get("opts") or []
    if not isinstance(opts, list):
        opts = []
    opts = [str(x) for x in opts]
    opts += list(MANUAL_COMMON_OPTS)
    opts += list(MANUAL_MMRL_OPTS if method_name == "MMRL" else MANUAL_BAYESRT_OPTS)

    # Force method/execution/shot after restored opts so method is correct.
    opts += [
        "METHOD.NAME", method_name,
        "METHOD.EXEC_MODE", "online",
        "DATASET.NUM_SHOTS", str(case["shot"]),
    ]

    args["opts"] = opts

    return Namespace(**args), log_path, parsed


def validate_cfg_ready(cfg, method_name, log_path=None):
    backbone = str(getattr(cfg.MODEL.BACKBONE, "NAME", ""))
    if not backbone:
        msg = [
            f"MODEL.BACKBONE.NAME is empty for method={method_name}.",
            "This means the notebook did not load the original runtime/method config.",
        ]
        if log_path is not None:
            msg.append(f"Parsed log: {log_path}")
        msg.append("Fix options:")
        msg.append("  1) set CASE['runtime_config_file'] / CASE['method_config_file_*'] to the original yaml files; or")
        msg.append("  2) set MANUAL_COMMON_OPTS = ['MODEL.BACKBONE.NAME', 'ViT-B/16'] plus any missing cfg opts.")
        raise ValueError("\\n".join(msg))


def build_loaded_trainer(case, method_name):
    if method_name == "MMRL":
        method_cfg = case.get("method_config_file_mmrl", "")
        model_dir = case["mmrl_model_dir"]
    elif method_name == "BayesRTMMRL":
        method_cfg = case.get("method_config_file_bayesrt", "")
        model_dir = case["bayesrt_model_dir"]
    else:
        raise ValueError(method_name)

    args, log_path, parsed = make_args_from_log_or_case(case, method_name, method_cfg, model_dir)
    print(f"[{method_name}] using log:", log_path)
    print(f"[{method_name}] dataset_config_file:", args.dataset_config_file)
    print(f"[{method_name}] method_config_file :", args.method_config_file)
    print(f"[{method_name}] runtime_config_file:", args.runtime_config_file)
    print(f"[{method_name}] protocol_config_file:", args.protocol_config_file)
    print(f"[{method_name}] exp_config         :", args.exp_config)
    print(f"[{method_name}] opts tail:", args.opts[-12:] if args.opts else [])

    cfg = setup_cfg(args)
    validate_cfg_ready(cfg, method_name, log_path=log_path)

    if cfg.SEED >= 0:
        set_random_seed(cfg.SEED)

    trainer = build_trainer(cfg)
    trainer.load_model(args.model_dir, epoch=args.load_epoch)
    trainer.model.eval()
    trainer.method.model.eval()
    return trainer


def get_loader(trainer, split):
    split = str(split).lower()
    if split == "test":
        candidates = ["test_loader", "test_loader_x"]
    elif split == "val":
        candidates = ["val_loader", "val_loader_x"]
    elif split in {"train", "train_x"}:
        candidates = ["train_loader_x", "train_loader"]
    else:
        candidates = [f"{split}_loader", f"{split}_loader_x"]

    for name in candidates:
        loader = getattr(trainer, name, None)
        if loader is not None:
            return loader

    dm = getattr(trainer, "dm", None)
    if dm is not None:
        for name in candidates:
            loader = getattr(dm, name, None)
            if loader is not None:
                return loader

    raise AttributeError(f"Cannot find loader for split={split}. Tried {candidates}")


def make_eval_ctx(cfg, split):
    dataset_name = str(getattr(cfg.DATASET, "NAME", CASE.get("dataset", "")))
    protocol = str(getattr(cfg.PROTOCOL, "NAME", CASE.get("protocol", "FS")))
    subsample_classes = getattr(cfg.DATASET, "SUBSAMPLE_CLASSES", None)
    phase = getattr(cfg.PROTOCOL, "PHASE", None)
    return EvalContext(
        protocol=protocol,
        dataset_name=dataset_name,
        split=str(split),
        subsample_classes=subsample_classes,
        phase=phase,
    )


def batch_to_device(batch, device):
    out = {}
    for k, v in batch.items():
        if torch.is_tensor(v):
            out[k] = v.to(device)
        else:
            out[k] = v
    return out


print("Building MMRL trainer...")
trainer_mmrl = build_loaded_trainer(CASE, "MMRL")
print("Building BayesRTMMRL trainer...")
trainer_bayes = build_loaded_trainer(CASE, "BayesRTMMRL")

split = CASE.get("split", "test")
loader_mmrl = get_loader(trainer_mmrl, split)
loader_bayes = get_loader(trainer_bayes, split)

eval_ctx_mmrl = make_eval_ctx(trainer_mmrl.cfg, split)
eval_ctx_bayes = make_eval_ctx(trainer_bayes.cfg, split)

n_mc = int(N_MC_OVERRIDE or getattr(trainer_bayes.method, "n_mc_test", trainer_bayes.cfg.BAYESRT_MMRL.N_MC_TEST))
print("split =", split)
print("n_mc =", n_mc)
print("eval_ctx_mmrl =", eval_ctx_mmrl)
print("eval_ctx_bayes =", eval_ctx_bayes)


Building MMRL trainer...
[MMRL] using log: /root/autodl-tmp/MMRL/output_refactor/MMRL/FS/fewshot_train/dtd/shots_16/ViT-B-16/default/seed1/run.log
[MMRL] dataset_config_file: /root/autodl-tmp/MMRL/configs/datasets/dtd.yaml
[MMRL] method_config_file : /root/autodl-tmp/MMRL/configs/methods/mmrl.yaml
[MMRL] runtime_config_file: /root/autodl-tmp/MMRL/configs/runtime/mmrl_family.yaml
[MMRL] protocol_config_file: /root/autodl-tmp/MMRL/configs/protocols/fs.yaml
[MMRL] exp_config         : 
[MMRL] opts tail: ['DATASET.NUM_SHOTS', '16', 'DATASET.SUBSAMPLE_CLASSES', 'all', 'MODEL.BACKBONE.NAME', 'ViT-B/16', 'METHOD.NAME', 'MMRL', 'METHOD.EXEC_MODE', 'online', 'DATASET.NUM_SHOTS', '16']
Loading trainer: RefactorRunner
Loading dataset: DescribableTextures
Reading split from /root/autodl-tmp/MMRL/DATASETS/dtd/split_zhou_DescribableTextures.json
Loading preprocessed few-shot data from /root/autodl-tmp/MMRL/DATASETS/dtd/split_fewshot/shot_16-seed_1.pkl
Building transform_train
+ random resized crop (

/root/autodl-tmp/MMRL/trainers/refactor_runner.py:74: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler() if prec == "amp" else None


[BayesRTMMRL] trainable params: {'representation_learner.compound_rep_tokens_r2vproj.4.bias', 'representation_learner.compound_rep_tokens_r2tproj.1.weight', 'representation_learner.compound_rep_tokens_r2vproj.6.bias', 'representation_learner.compound_rep_tokens_r2tproj.2.bias', 'representation_learner.compound_rep_tokens_r2tproj.4.weight', 'representation_learner.compound_rep_tokens_r2vproj.3.weight', 'representation_learner.compound_rep_tokens', 'representation_learner.compound_rep_tokens_r2vproj.0.weight', 'representation_learner.compound_rep_tokens_r2tproj.1.bias', 'representation_learner.compound_rep_tokens_r2vproj.1.bias', 'representation_learner.compound_rep_tokens_r2vproj.0.bias', 'representation_learner.compound_rep_tokens_r2vproj.2.bias', 'text_posterior.posterior_rho', 'representation_learner.compound_rep_tokens_r2tproj.5.weight', 'image_encoder.bayes_proj_rep.posterior_rho', 'representation_learner.compound_rep_tokens_r2vproj.1.weight', 'representation_learner.compound_rep_t

In [4]:
# =========================
# 3. metric / calibration 工具
# =========================
def to_numpy(x):
    if x is None:
        return None
    if torch.is_tensor(x):
        return x.detach().float().cpu().numpy()
    return np.asarray(x)


def label_to_numpy(x):
    if x is None:
        return None
    if torch.is_tensor(x):
        return x.detach().cpu().numpy().astype(np.int64)
    return np.asarray(x).astype(np.int64)


def softmax_np(z):
    z = np.asarray(z, dtype=np.float64)
    z = z - np.max(z, axis=1, keepdims=True)
    e = np.exp(z)
    return e / np.clip(e.sum(axis=1, keepdims=True), 1e-12, None)


def entropy_from_probs_np(p):
    p = np.clip(np.asarray(p, dtype=np.float64), 1e-12, 1.0)
    h = -(p * np.log(p)).sum(axis=1)
    k = p.shape[1]
    if k > 1:
        h = h / np.log(k)
    return h


def ece_score(probs, y, n_bins=15, adaptive=False):
    probs = np.asarray(probs, dtype=np.float64)
    y = np.asarray(y).astype(int)
    conf = probs.max(axis=1)
    pred = probs.argmax(axis=1)
    corr = (pred == y).astype(float)

    if adaptive:
        edges = np.quantile(conf, np.linspace(0, 1, n_bins + 1))
        edges[0], edges[-1] = 0.0, 1.0
        edges = np.unique(edges)
        if len(edges) <= 2:
            return 0.0
    else:
        edges = np.linspace(0, 1, n_bins + 1)

    ece = 0.0
    for i, (lo, hi) in enumerate(zip(edges[:-1], edges[1:])):
        mask = (conf >= lo) & (conf <= hi) if i == len(edges) - 2 else (conf >= lo) & (conf < hi)
        if not np.any(mask):
            continue
        ece += mask.mean() * abs(corr[mask].mean() - conf[mask].mean())
    return float(ece)


def metric_row(logits, y, name=None):
    z = np.asarray(logits, dtype=np.float64)
    y = np.asarray(y).astype(int)
    if z.shape[0] != len(y):
        raise ValueError(f"{name or 'logits'} rows {z.shape[0]} != labels rows {len(y)}")

    p = softmax_np(z)
    pred = p.argmax(axis=1)
    conf = p.max(axis=1)
    corr = pred == y

    nll = -np.log(np.clip(p[np.arange(len(y)), y], 1e-12, 1.0)).mean()
    onehot = np.zeros_like(p)
    onehot[np.arange(len(y)), y] = 1.0
    brier = ((p - onehot) ** 2).sum(axis=1).mean()

    z_masked = z.copy()
    z_masked[np.arange(len(y)), y] = -np.inf
    true_margin = z[np.arange(len(y)), y] - np.max(z_masked, axis=1)
    part = np.partition(z, -2, axis=1)
    top2_margin = part[:, -1] - part[:, -2]

    out = {
        "n": int(len(y)),
        "acc": float(corr.mean()),
        "ece": ece_score(p, y, n_bins=N_BINS, adaptive=False),
        "ece_adaptive": ece_score(p, y, n_bins=N_BINS, adaptive=True),
        "nll": float(nll),
        "brier": float(brier),
        "confidence_mean": float(conf.mean()),
        "entropy_mean": float(entropy_from_probs_np(p).mean()),
        "calibration_gap_conf_minus_acc": float(conf.mean() - corr.mean()),
        "true_margin_mean": float(true_margin.mean()),
        "top2_margin_mean": float(top2_margin.mean()),
    }

    for t in HC_THRESHOLDS:
        wrong = ~corr
        tag = str(t).replace(".", "p")
        out[f"wrong_conf_ge_{tag}"] = int(((conf >= t) & wrong).sum())
        out[f"wrong_conf_ge_{tag}_rate_over_wrong"] = float(((conf >= t) & wrong).sum() / max(1, wrong.sum()))

    return out


def reliability_bins_df(logits, y, candidate, case_name, n_bins=15):
    z = np.asarray(logits, dtype=np.float64)
    p = softmax_np(z)
    y = np.asarray(y).astype(int)
    conf = p.max(axis=1)
    pred = p.argmax(axis=1)
    corr = (pred == y).astype(float)
    edges = np.linspace(0, 1, n_bins + 1)
    rows = []
    for i, (lo, hi) in enumerate(zip(edges[:-1], edges[1:])):
        mask = (conf >= lo) & (conf <= hi) if i == n_bins - 1 else (conf >= lo) & (conf < hi)
        n = int(mask.sum())
        if n == 0:
            rows.append({
                "case_name": case_name, "candidate": candidate, "bin_id": i,
                "conf_lo": lo, "conf_hi": hi, "n": 0, "frac": 0.0,
                "acc": np.nan, "conf": np.nan, "gap": np.nan,
                "signed_gap_acc_minus_conf": np.nan, "ece_contribution": 0.0,
            })
        else:
            acc_b = corr[mask].mean()
            conf_b = conf[mask].mean()
            gap = abs(acc_b - conf_b)
            rows.append({
                "case_name": case_name, "candidate": candidate, "bin_id": i,
                "conf_lo": lo, "conf_hi": hi, "n": n, "frac": float(mask.mean()),
                "acc": float(acc_b), "conf": float(conf_b),
                "gap": float(gap),
                "signed_gap_acc_minus_conf": float(acc_b - conf_b),
                "ece_contribution": float(mask.mean() * gap),
            })
    return pd.DataFrame(rows)


def paired_outcome(base_logits, cand_logits, y):
    y = np.asarray(y).astype(int)
    pb = softmax_np(base_logits).argmax(axis=1)
    pc = softmax_np(cand_logits).argmax(axis=1)
    bc = pb == y
    cc = pc == y
    return {
        "both_correct": bc & cc,
        "harmful_flip": bc & (~cc),
        "beneficial_flip": (~bc) & cc,
        "both_wrong": (~bc) & (~cc),
    }


def paired_outcome_summary(base_logits, cand_logits, y, base_name, cand_name, case_name):
    masks = paired_outcome(base_logits, cand_logits, y)
    row = {"case_name": case_name, "base": base_name, "candidate": cand_name}
    for k, m in masks.items():
        row[k] = int(m.sum())
    row["net_gain"] = row["beneficial_flip"] - row["harmful_flip"]
    row["base_acc"] = float((softmax_np(base_logits).argmax(axis=1) == y).mean())
    row["candidate_acc"] = float((softmax_np(cand_logits).argmax(axis=1) == y).mean())
    row["delta_acc"] = row["candidate_acc"] - row["base_acc"]
    return row


In [ ]:
# =========================
# 4. MC uncertainty / disagreement 工具
# =========================
def mc_uncertainty_rows(logits_stack, y, candidate, case_name):
    # logits_stack shape: [S, N, C]
    z = np.asarray(logits_stack, dtype=np.float64)
    y = np.asarray(y).astype(np.int64)
    s, n, c = z.shape

    z_shift = z - np.max(z, axis=2, keepdims=True)
    p_stack = np.exp(z_shift)
    p_stack = p_stack / np.clip(p_stack.sum(axis=2, keepdims=True), 1e-12, None)

    p_mean = p_stack.mean(axis=0)
    pred_stack = p_stack.argmax(axis=2)
    pred_mean = p_mean.argmax(axis=1)

    vote_counts = np.zeros((n, c), dtype=np.float64)
    for i in range(n):
        vote_counts[i] = np.bincount(pred_stack[:, i], minlength=c)
    vote_probs = vote_counts / float(s)
    mode_frac = vote_probs.max(axis=1)

    sample_agreement = mode_frac
    variation_ratio = 1.0 - mode_frac
    vote_entropy = entropy_from_probs_np(vote_probs)

    predictive_entropy = entropy_from_probs_np(p_mean)
    expected_entropy = entropy_from_probs_np(p_stack.reshape(-1, c)).reshape(s, n).mean(axis=0)
    mutual_information = predictive_entropy - expected_entropy

    conf_stack = p_stack.max(axis=2)
    confidence_variance = conf_stack.var(axis=0)

    margins = []
    for ss in range(s):
        zz = z[ss].copy()
        zz[np.arange(n), y] = -np.inf
        max_wrong = zz.max(axis=1)
        margins.append(z[ss, np.arange(n), y] - max_wrong)
    margins = np.stack(margins, axis=0)
    margin_variance = margins.var(axis=0)

    correct = pred_mean == y

    df = pd.DataFrame({
        "case_name": case_name,
        "candidate": candidate,
        "sample_agreement": sample_agreement,
        "variation_ratio": variation_ratio,
        "vote_entropy": vote_entropy,
        "predictive_entropy": predictive_entropy,
        "expected_entropy": expected_entropy,
        "mutual_information": mutual_information,
        "confidence_variance": confidence_variance,
        "margin_variance": margin_variance,
        "mean_pred_correct": correct.astype(int),
    })

    summary = {
        "case_name": case_name,
        "candidate": candidate,
        "sample_agreement_mean": float(sample_agreement.mean()),
        "variation_ratio_mean": float(variation_ratio.mean()),
        "vote_entropy_mean": float(vote_entropy.mean()),
        "predictive_entropy_mean": float(predictive_entropy.mean()),
        "expected_entropy_mean": float(expected_entropy.mean()),
        "mutual_information_mean": float(mutual_information.mean()),
        "confidence_variance_mean": float(confidence_variance.mean()),
        "margin_variance_mean": float(margin_variance.mean()),
    }

    for group_name, mask in {"correct": correct, "wrong": ~correct}.items():
        if np.any(mask):
            summary[f"{group_name}_sample_agreement_mean"] = float(sample_agreement[mask].mean())
            summary[f"{group_name}_variation_ratio_mean"] = float(variation_ratio[mask].mean())
            summary[f"{group_name}_mi_mean"] = float(mutual_information[mask].mean())
            summary[f"{group_name}_predictive_entropy_mean"] = float(predictive_entropy[mask].mean())

    return df, summary


In [ ]:
# =========================
# 5. MMRL official output 收集
# =========================
@torch.no_grad()
def collect_mmrl_outputs(trainer, loader, eval_ctx):
    method = trainer.method
    method.model.eval()

    labels = []
    logits = {"C": [], "R": [], "fusion": [], "reported": []}

    for batch in loader:
        batch = batch_to_device(batch, trainer.device)
        out = method.forward_eval(batch, eval_ctx)
        y = out.labels
        rep = out.aux_logits.get("rep")
        fusion = out.aux_logits.get("fusion")
        reported = method.select_eval_logits(out, eval_ctx)

        labels.append(label_to_numpy(y))
        logits["C"].append(to_numpy(out.logits))
        logits["R"].append(to_numpy(rep))
        logits["fusion"].append(to_numpy(fusion))
        logits["reported"].append(to_numpy(reported))

    labels = np.concatenate(labels, axis=0)
    logits = {k: np.concatenate(v, axis=0) for k, v in logits.items()}
    return {"labels": labels, "logits": logits}


mmrl_data = collect_mmrl_outputs(trainer_mmrl, loader_mmrl, eval_ctx_mmrl)
print("MMRL labels:", mmrl_data["labels"].shape)
for k, v in mmrl_data["logits"].items():
    print("MMRL", k, v.shape)


In [ ]:
# =========================
# 6. BayesRT split forward：T mean/MC 与 R mean/MC 独立拆分
# =========================
@torch.no_grad()
def bayesrt_components_for_batch(method, image, num_samples):
    model = method.model
    model.eval()

    compound_rep_tokens_text, compound_rep_tokens_visual = model.representation_learner()

    eot_hidden = model.text_encoder.forward_hidden(
        model.prompt_embeddings,
        model.tokenized_prompts,
        compound_rep_tokens_text,
    )

    text_mc, _ = model.text_sample_features(
        eot_hidden=eot_hidden,
        num_samples=num_samples,
        use_mean=False,
    )
    text_mean_stack, _ = model.text_sample_features(
        eot_hidden=eot_hidden,
        num_samples=num_samples,
        use_mean=True,
    )

    image_main, image_rep_mean, _, rep_hidden = model.image_encoder.forward_mean(
        image.type(model.dtype),
        compound_rep_tokens_visual,
    )

    rep_mc = model.image_encoder.rep_samples(
        rep_hidden=rep_hidden,
        num_samples=num_samples,
        use_mean=False,
    )
    rep_mean_stack = model.image_encoder.rep_samples(
        rep_hidden=rep_hidden,
        num_samples=num_samples,
        use_mean=True,
    )

    image_main = F.normalize(image_main, dim=-1)
    rep_mc = F.normalize(rep_mc, dim=-1)
    rep_mean_stack = F.normalize(rep_mean_stack, dim=-1)

    text_mc = text_mc.type(image_main.dtype)
    text_mean_stack = text_mean_stack.type(image_main.dtype)

    return {
        "image_main": image_main,
        "text": {"Tmc": text_mc, "Tmean": text_mean_stack},
        "rep": {"Rmc": rep_mc, "Rmean": rep_mean_stack},
    }


def build_logits_from_components(components, alpha):
    image_main = components["image_main"]
    out = {}

    for t_name, text_samples in components["text"].items():
        logits_main_stack = 100.0 * torch.einsum("bd,scd->sbc", image_main, text_samples)
        out[f"C_{t_name}"] = logits_main_stack

        for r_name, rep_stack in components["rep"].items():
            logits_rep_stack = 100.0 * torch.einsum("sbd,scd->sbc", rep_stack, text_samples)
            out[f"R_{r_name}_{t_name}"] = logits_rep_stack

            p_c = torch.softmax(logits_main_stack.float(), dim=-1).mean(dim=0)
            p_r = torch.softmax(logits_rep_stack.float(), dim=-1).mean(dim=0)
            p_f = alpha * p_c + (1.0 - alpha) * p_r
            log_f = torch.log(p_f.clamp_min(1.0e-12)).to(logits_main_stack.dtype)
            out[f"fusion_static_prob_{r_name}_{t_name}"] = log_f.unsqueeze(0).expand_as(logits_main_stack)

    return out


def aggregate_prob_mean_stack(logits_stack):
    p = torch.softmax(logits_stack.float(), dim=-1).mean(dim=0)
    return torch.log(p.clamp_min(1.0e-12)).to(logits_stack.dtype)


@torch.no_grad()
def collect_bayesrt_split_outputs(trainer, loader, eval_ctx, num_samples):
    method = trainer.method
    method.model.eval()
    alpha = float(trainer.cfg.BAYESRT_MMRL.ALPHA)

    labels = []
    stacks = {}
    logits = {}

    for batch_idx, batch in enumerate(loader):
        batch = batch_to_device(batch, trainer.device)
        image = batch["img"].to(trainer.device)
        y = batch["label"].to(trainer.device)

        comps = bayesrt_components_for_batch(method, image, num_samples)
        batch_stacks = build_logits_from_components(comps, alpha=alpha)

        labels.append(label_to_numpy(y))

        for name, stack in batch_stacks.items():
            stack_np = to_numpy(stack)
            stacks.setdefault(name, []).append(stack_np)

            agg = aggregate_prob_mean_stack(stack)
            logits.setdefault(name, []).append(to_numpy(agg))

    labels = np.concatenate(labels, axis=0)
    stacks = {k: np.concatenate(v, axis=1) for k, v in stacks.items()}
    logits = {k: np.concatenate(v, axis=0) for k, v in logits.items()}

    return {"labels": labels, "logits": logits, "stacks": stacks}


bayes_split = collect_bayesrt_split_outputs(trainer_bayes, loader_bayes, eval_ctx_bayes, n_mc)
print("Bayes labels:", bayes_split["labels"].shape)
for k, v in bayes_split["logits"].items():
    print(k, v.shape, "stack", bayes_split["stacks"][k].shape)

assert np.array_equal(mmrl_data["labels"], bayes_split["labels"]), "MMRL/Bayes labels mismatch"
y = bayes_split["labels"]


In [ ]:
# =========================
# 7. 核心比较：MMRL C vs BayesRT C_Tmean vs BayesRT C_Tmc
# =========================
case_name = CASE["case_name"]

candidate_logits = {
    "MMRL_C": mmrl_data["logits"]["C"],
    "MMRL_R": mmrl_data["logits"]["R"],
    "MMRL_fusion_reported": mmrl_data["logits"]["reported"],

    "BayesRT_C_Tmean": bayes_split["logits"]["C_Tmean"],
    "BayesRT_C_Tmc": bayes_split["logits"]["C_Tmc"],

    "BayesRT_R_Rmean_Tmean": bayes_split["logits"]["R_Rmean_Tmean"],
    "BayesRT_R_Rmc_Tmean": bayes_split["logits"]["R_Rmc_Tmean"],
    "BayesRT_R_Rmean_Tmc": bayes_split["logits"]["R_Rmean_Tmc"],
    "BayesRT_R_Rmc_Tmc": bayes_split["logits"]["R_Rmc_Tmc"],

    "BayesRT_fusion_Rmean_Tmean": bayes_split["logits"]["fusion_static_prob_Rmean_Tmean"],
    "BayesRT_fusion_Rmc_Tmean": bayes_split["logits"]["fusion_static_prob_Rmc_Tmean"],
    "BayesRT_fusion_Rmean_Tmc": bayes_split["logits"]["fusion_static_prob_Rmean_Tmc"],
    "BayesRT_fusion_Rmc_Tmc": bayes_split["logits"]["fusion_static_prob_Rmc_Tmc"],
}

metrics_rows = []
for name, z in candidate_logits.items():
    row = {"case_name": case_name, "candidate": name}
    row.update(metric_row(z, y, name=name))
    metrics_rows.append(row)

metrics_df = pd.DataFrame(metrics_rows).sort_values(["ece", "nll", "brier"])
display(metrics_df)

delta_rows = []
for a_name, b_name in [
    ("MMRL_C", "BayesRT_C_Tmean"),
    ("BayesRT_C_Tmean", "BayesRT_C_Tmc"),
    ("MMRL_C", "BayesRT_C_Tmc"),
]:
    a = metrics_df[metrics_df["candidate"] == a_name].iloc[0]
    b = metrics_df[metrics_df["candidate"] == b_name].iloc[0]
    row = {"case_name": case_name, "from": a_name, "to": b_name}
    for col in ["acc", "ece", "ece_adaptive", "nll", "brier", "confidence_mean", "entropy_mean", "true_margin_mean", "top2_margin_mean"]:
        row[f"delta_{col}"] = float(b[col] - a[col])
    delta_rows.append(row)

c_delta_df = pd.DataFrame(delta_rows)
display(c_delta_df)


In [ ]:
# =========================
# 8. MC disagreement：T MC 是否真的产生不确定性？
# =========================
mc_summary_rows = []
mc_sample_dfs = []

for name in ["C_Tmc", "C_Tmean", "R_Rmc_Tmc", "R_Rmean_Tmc", "R_Rmc_Tmean", "fusion_static_prob_Rmc_Tmc"]:
    if name not in bayes_split["stacks"]:
        continue
    df_sample, summary = mc_uncertainty_rows(
        bayes_split["stacks"][name],
        y,
        candidate=f"BayesRT_{name}",
        case_name=case_name,
    )
    mc_sample_dfs.append(df_sample)
    mc_summary_rows.append(summary)

mc_summary_df = pd.DataFrame(mc_summary_rows).sort_values(["mutual_information_mean", "variation_ratio_mean"])
display(mc_summary_df)

mc_sample_df = pd.concat(mc_sample_dfs, ignore_index=True)


In [ ]:
# =========================
# 9. Reliability bin 分解：ECE 从哪里来？
# =========================
rel_candidates = [
    "MMRL_C",
    "BayesRT_C_Tmean",
    "BayesRT_C_Tmc",
    "MMRL_fusion_reported",
    "BayesRT_fusion_Rmc_Tmc",
]

rel_dfs = []
for name in rel_candidates:
    rel_dfs.append(reliability_bins_df(candidate_logits[name], y, name, case_name, n_bins=N_BINS))

reliability_df = pd.concat(rel_dfs, ignore_index=True)
display(
    reliability_df
    .sort_values(["candidate", "ece_contribution"], ascending=[True, False])
    .groupby("candidate")
    .head(5)
)

high_bin_df = (
    reliability_df[reliability_df["conf_lo"] >= 0.8]
    .groupby(["case_name", "candidate"], as_index=False)
    .agg(
        high_bin_n=("n", "sum"),
        high_bin_frac=("frac", "sum"),
        high_bin_ece_contribution=("ece_contribution", "sum"),
    )
)
total_ece = reliability_df.groupby(["case_name", "candidate"], as_index=False)["ece_contribution"].sum()
high_bin_df = high_bin_df.merge(total_ece, on=["case_name", "candidate"], how="left", suffixes=("", "_total"))
high_bin_df["high_bin_share_of_ece"] = high_bin_df["high_bin_ece_contribution"] / high_bin_df["ece_contribution"].replace(0, np.nan)
display(high_bin_df.sort_values("high_bin_ece_contribution", ascending=False))


In [ ]:
# =========================
# 10. Paired outcome 分析：C_Tmean / C_Tmc 对 MMRL C 的影响
# =========================
paired_rows = []
for cand in ["BayesRT_C_Tmean", "BayesRT_C_Tmc", "BayesRT_fusion_Rmean_Tmean", "BayesRT_fusion_Rmc_Tmc"]:
    paired_rows.append(
        paired_outcome_summary(
            candidate_logits["MMRL_C"],
            candidate_logits[cand],
            y,
            base_name="MMRL_C",
            cand_name=cand,
            case_name=case_name,
        )
    )
paired_df = pd.DataFrame(paired_rows)
display(paired_df)

def per_outcome_shift_df(base_logits, cand_logits, y, base_name, cand_name):
    yy = np.asarray(y).astype(np.int64)
    masks = paired_outcome(base_logits, cand_logits, yy)
    p_base = softmax_np(base_logits)
    p_cand = softmax_np(cand_logits)

    conf_base = p_base.max(axis=1)
    conf_cand = p_cand.max(axis=1)
    ent_base = entropy_from_probs_np(p_base)
    ent_cand = entropy_from_probs_np(p_cand)

    def true_margin_np(z):
        z = np.asarray(z, dtype=np.float64)
        z_masked = z.copy()
        z_masked[np.arange(len(yy)), yy] = -np.inf
        return z[np.arange(len(yy)), yy] - z_masked.max(axis=1)

    def top2_margin_np(z):
        z = np.asarray(z, dtype=np.float64)
        part = np.partition(z, -2, axis=1)
        return part[:, -1] - part[:, -2]

    tm_base = true_margin_np(base_logits)
    tm_cand = true_margin_np(cand_logits)
    top2_base = top2_margin_np(base_logits)
    top2_cand = top2_margin_np(cand_logits)

    rows = []
    for outcome, mask in masks.items():
        if not np.any(mask):
            continue
        rows.append({
            "case_name": case_name,
            "base": base_name,
            "candidate": cand_name,
            "outcome": outcome,
            "n": int(mask.sum()),
            "base_conf": float(conf_base[mask].mean()),
            "cand_conf": float(conf_cand[mask].mean()),
            "delta_conf": float((conf_cand - conf_base)[mask].mean()),
            "base_entropy": float(ent_base[mask].mean()),
            "cand_entropy": float(ent_cand[mask].mean()),
            "delta_entropy": float((ent_cand - ent_base)[mask].mean()),
            "base_true_margin": float(tm_base[mask].mean()),
            "cand_true_margin": float(tm_cand[mask].mean()),
            "delta_true_margin": float((tm_cand - tm_base)[mask].mean()),
            "base_top2_margin": float(top2_base[mask].mean()),
            "cand_top2_margin": float(top2_cand[mask].mean()),
            "delta_top2_margin": float((top2_cand - top2_base)[mask].mean()),
        })
    return pd.DataFrame(rows)

outcome_shift_dfs = []
for cand in ["BayesRT_C_Tmean", "BayesRT_C_Tmc"]:
    outcome_shift_dfs.append(
        per_outcome_shift_df(
            candidate_logits["MMRL_C"],
            candidate_logits[cand],
            y,
            base_name="MMRL_C",
            cand_name=cand,
        )
    )
outcome_shift_df = pd.concat(outcome_shift_dfs, ignore_index=True)
display(outcome_shift_df)


In [ ]:
# =========================
# 11. 样本级诊断
# =========================
def sample_diag_for_candidate(logits, y, prefix):
    yy = np.asarray(y).astype(np.int64)
    p = softmax_np(logits)
    pred = p.argmax(axis=1)
    conf = p.max(axis=1)
    ent = entropy_from_probs_np(p)

    z = np.asarray(logits, dtype=np.float64)
    z_masked = z.copy()
    z_masked[np.arange(len(yy)), yy] = -np.inf
    true_margin = z[np.arange(len(yy)), yy] - z_masked.max(axis=1)
    part = np.partition(z, -2, axis=1)
    top2_margin = part[:, -1] - part[:, -2]

    return pd.DataFrame({
        f"{prefix}_pred": pred,
        f"{prefix}_correct": (pred == yy).astype(int),
        f"{prefix}_conf": conf,
        f"{prefix}_entropy": ent,
        f"{prefix}_true_margin": true_margin,
        f"{prefix}_top2_margin": top2_margin,
    })

sample_df = pd.DataFrame({
    "case_name": case_name,
    "sample_index": np.arange(len(y)),
    "label": y,
})

for prefix, logits in [
    ("MMRL_C", candidate_logits["MMRL_C"]),
    ("BayesRT_C_Tmean", candidate_logits["BayesRT_C_Tmean"]),
    ("BayesRT_C_Tmc", candidate_logits["BayesRT_C_Tmc"]),
    ("MMRL_reported", candidate_logits["MMRL_fusion_reported"]),
    ("BayesRT_fusion_Rmc_Tmc", candidate_logits["BayesRT_fusion_Rmc_Tmc"]),
]:
    sample_df = pd.concat([sample_df, sample_diag_for_candidate(logits, y, prefix)], axis=1)

c_mc_unc = mc_sample_df[mc_sample_df["candidate"] == "BayesRT_C_Tmc"].reset_index(drop=True)
for col in ["sample_agreement", "variation_ratio", "vote_entropy", "predictive_entropy",
            "expected_entropy", "mutual_information", "confidence_variance", "margin_variance"]:
    sample_df[f"C_Tmc_{col}"] = c_mc_unc[col].values

sample_df["delta_conf_Tmc_vs_Tmean"] = sample_df["BayesRT_C_Tmc_conf"] - sample_df["BayesRT_C_Tmean_conf"]
sample_df["delta_entropy_Tmc_vs_Tmean"] = sample_df["BayesRT_C_Tmc_entropy"] - sample_df["BayesRT_C_Tmean_entropy"]
sample_df["delta_top2_margin_Tmc_vs_Tmean"] = sample_df["BayesRT_C_Tmc_top2_margin"] - sample_df["BayesRT_C_Tmean_top2_margin"]

sample_df["delta_conf_BayesTmc_vs_MMRLC"] = sample_df["BayesRT_C_Tmc_conf"] - sample_df["MMRL_C_conf"]
sample_df["delta_entropy_BayesTmc_vs_MMRLC"] = sample_df["BayesRT_C_Tmc_entropy"] - sample_df["MMRL_C_entropy"]
sample_df["delta_top2_margin_BayesTmc_vs_MMRLC"] = sample_df["BayesRT_C_Tmc_top2_margin"] - sample_df["MMRL_C_top2_margin"]

for t in HC_THRESHOLDS:
    suffix = str(t).replace(".", "p")
    sample_df[f"BayesRT_C_Tmc_wrong_conf_ge_{suffix}"] = (
        (sample_df["BayesRT_C_Tmc_correct"] == 0) & (sample_df["BayesRT_C_Tmc_conf"] >= t)
    ).astype(int)
    sample_df[f"MMRL_C_wrong_conf_ge_{suffix}"] = (
        (sample_df["MMRL_C_correct"] == 0) & (sample_df["MMRL_C_conf"] >= t)
    ).astype(int)

display(sample_df.head())


In [ ]:
# =========================
# 12. 自动判读：观察、代码机制、未验证项分开
# =========================
def get_metric(name):
    return metrics_df[metrics_df["candidate"] == name].iloc[0].to_dict()

mmrl_c_m = get_metric("MMRL_C")
c_tmean_m = get_metric("BayesRT_C_Tmean")
c_tmc_m = get_metric("BayesRT_C_Tmc")

judgement = []

judgement.append({
    "question": "BayesRT C overconfidence 是否已经在 posterior mean path 存在？",
    "evidence": f"C_Tmean ECE={c_tmean_m['ece']:.4f}, conf={c_tmean_m['confidence_mean']:.4f}; "
                f"MMRL_C ECE={mmrl_c_m['ece']:.4f}, conf={mmrl_c_m['confidence_mean']:.4f}",
    "interpretation": (
        "如果 C_Tmean 已明显高于 MMRL_C 的 ECE/confidence，则问题不是 MC sampling alone，"
        "而是 BayesRT 训练后的 mean path 已经 sharpened。"
    ),
    "result": "C_Tmean_overconfident" if c_tmean_m["ece"] > mmrl_c_m["ece"] and c_tmean_m["confidence_mean"] > mmrl_c_m["confidence_mean"] else "not_supported",
})

judgement.append({
    "question": "T MC sampling 是否进一步恶化 C calibration？",
    "evidence": f"C_Tmc - C_Tmean: delta_ECE={c_tmc_m['ece'] - c_tmean_m['ece']:+.4f}, "
                f"delta_conf={c_tmc_m['confidence_mean'] - c_tmean_m['confidence_mean']:+.4f}, "
                f"delta_entropy={c_tmc_m['entropy_mean'] - c_tmean_m['entropy_mean']:+.4f}",
    "interpretation": (
        "若 Tmc 相比 Tmean ECE/confidence 上升且 entropy 下降，说明 T MC 在 eval 端进一步 sharpening；"
        "若 Tmc ECE 下降，则 MC 具有 calibration value。"
    ),
    "result": (
        "T_MC_worsens_calibration"
        if c_tmc_m["ece"] > c_tmean_m["ece"] and c_tmc_m["confidence_mean"] > c_tmean_m["confidence_mean"]
        else "T_MC_not_worsening_or_improves"
    ),
})

c_unc = mc_summary_df[mc_summary_df["candidate"] == "BayesRT_C_Tmc"].iloc[0].to_dict()
judgement.append({
    "question": "T MC 是否产生足够 predictive disagreement？",
    "evidence": f"C_Tmc sample_agreement_mean={c_unc['sample_agreement_mean']:.4f}, "
                f"variation_ratio_mean={c_unc['variation_ratio_mean']:.4f}, "
                f"MI_mean={c_unc['mutual_information_mean']:.4f}",
    "interpretation": (
        "若 sample_agreement 接近 1、variation/MI 很低，则 T posterior 没有提供有效类别分歧，"
        "BMA 不会显著降低 confidence。"
    ),
    "result": (
        "low_disagreement"
        if c_unc["sample_agreement_mean"] > 0.95 and c_unc["mutual_information_mean"] < 0.02
        else "nontrivial_disagreement"
    ),
})

judgement_df = pd.DataFrame(judgement)
display(judgement_df)


In [ ]:
# =========================
# 13. 保存结果
# =========================
xlsx_path = OUT_DIR / f"{case_name}_c_branch_sharpening_mechanism.xlsx"
csv_sample_path = OUT_DIR / f"{case_name}_c_branch_sample_diagnostics.csv"

with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    metrics_df.to_excel(writer, sheet_name="CoreMetrics", index=False)
    c_delta_df.to_excel(writer, sheet_name="CoreDeltas", index=False)
    mc_summary_df.to_excel(writer, sheet_name="MCDisagreementSummary", index=False)
    reliability_df.to_excel(writer, sheet_name="ReliabilityBins", index=False)
    high_bin_df.to_excel(writer, sheet_name="HighConfBinSummary", index=False)
    paired_df.to_excel(writer, sheet_name="PairedOutcome", index=False)
    outcome_shift_df.to_excel(writer, sheet_name="OutcomeShifts", index=False)
    judgement_df.to_excel(writer, sheet_name="MechanismJudgement", index=False)

if SAVE_SAMPLE_DIAGNOSTICS:
    sample_df.to_csv(csv_sample_path, index=False)
    print("Saved sample diagnostics:", csv_sample_path)

print("Saved Excel:", xlsx_path)
display(judgement_df)


## 判读标准

### 1. `CoreMetrics`

比较：

- `MMRL_C`
- `BayesRT_C_Tmean`
- `BayesRT_C_Tmc`

判断：

```text
如果 BayesRT_C_Tmean 已经比 MMRL_C 更高 confidence / ECE：
  mean path 已经 sharpened，不能把问题单独归因于 MC sampling。

如果 BayesRT_C_Tmc 比 BayesRT_C_Tmean 更高 confidence / ECE：
  T MC sampling 进一步恶化 calibration。

如果 BayesRT_C_Tmc 比 BayesRT_C_Tmean 更低 ECE：
  MC sampling 有 calibration value，问题主要在 mean path。
```

### 2. `MCDisagreementSummary`

判断 T posterior 是否真的产生不确定性：

```text
sample_agreement_mean 接近 1
variation_ratio_mean 很低
mutual_information_mean 很低
```

说明 MC samples 基本预测同一类别，Bayesian averaging 不会显著 softening。

### 3. `ReliabilityBins`

看 high confidence bins 是否吸收了太多样本：

```text
BayesRT_C_Tmc 在 0.8-1.0 bins 的 n / ece_contribution 是否显著高于 MMRL_C。
```

### 4. `OutcomeShifts`

看 `both_wrong`：

```text
both_wrong 的 delta_conf 是否 > 0
both_wrong 的 delta_entropy 是否 < 0
both_wrong 的 delta_true_margin 是否更负
```

如果成立，说明 BayesRT C 对仍然错误的样本也 sharpened，这是 ECE 上升的根本原因。
